In [1]:
import pandas as pd
import os
from collections import Counter
import re
from nltk.corpus import stopwords
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import joblib
from sklearn.model_selection import RandomizedSearchCV
from pprint import pprint

In [2]:
# Download stopwords if not already present
try:
    nltk.data.find('corpora/stopwords')
except nltk.downloader.DownloadError:
    nltk.download('stopwords')

 ### Load and Display Data

In [5]:
# Load the datasets
try:
    # NOTE: Update these paths to your actual file locations
    train_df = pd.read_csv(r'C:\Users\ADMIN\return_risk\data\raw\training_data_womensUK.csv', encoding='latin1', on_bad_lines='skip')
    test_df = pd.read_csv(r'C:\Users\ADMIN\return_risk\data\raw\test_data_womensUK.csv', encoding='latin1', on_bad_lines='skip')
    print("Training and test data loaded successfully.")
except FileNotFoundError as e:
    print(e)
    print("\nPlease make sure the files 'training_data_womensUK.csv' and 'test_data_womensUK.csv' are in the same directory.")

Training and test data loaded successfully.


In [6]:
# Display initial information
print("\nTraining Data Head:")
display(train_df.head())
print("\nTest Data Head:")
display(test_df.head())
print("\nTraining Data Info:")
train_df.info()
print("\nTest Data Info:")
test_df.info()


Training Data Head:


,Product ID,Name,Retailer,Brand,Segment,Gender,Category,Color,Activewear,Pattern,...,Current Discount Percentage,Date First Discounted,SKUs Available,Num Replenishments,Days to Majority SKU sellout,Days to First sellout,Season,Description,Care information,Sizes
0,dc10ba7115fc8e45c03ab71a7e8fb67a24546d1faf874d...,Black Lotty Sliders,TK Maxx (UK),Replay,Mass,Women,Footwear,Black,Non-activewear,Product has no pattern data,...,0.0%,NaN,1/1,0,NaN,NaN,ss25,Sliders\nBlack\nLotty style\nReptile effect fr...,Upper: Synthetic\nInner: Synthetic\nSole: Synt...,4
1,46a57ebe0ad93bcf2c613ec304c31fc16f73f6b3d41081...,Kensey' Casual Lightweight Trainers,Debenhams (UK),Moshulu,Mass,Women,Footwear,Pink,Non-activewear,Product has no pattern data,...,0.0%,NaN,6/7,0,NaN,NaN,aw25,Kensey makes every step comfier! In smooth lea...,"Upper: Nubuck (Indigo, Light Green) or Leather...","3, 4, 5, 6, 6.5, 7, 8"
2,e05afa3109a21743ebd9f7ac0fbb1ce218caa82f241e94...,Kalisa Bow Top in Textured Pink,Motel (UK),Motel,Mass,Women,Tops,Pink,Non-activewear,Plain,...,0.0%,3 Jul 2025,3/7,0,173.0,NaN,ss25,The Kalisa top\nin a pink textured material\nF...,NaN,"XXS, XS, S, M, L, XL, XXL"
3,03b218b35f7afc4d29514f2133af648e33dcbd7d93e557...,2000-2015 Gg Canvas Bamboo Handbag,Farfetch (UK),Gucci Pre-Owned,Luxury,Women,Accessories,Brown,Non-activewear,Tile,...,0.0%,15 Jun 2025,1/1,0,NaN,NaN,ss25,Pre-Owned\n2000-2015 GG Canvas Bamboo handbag\...,Outer:\nCanvas 100%\nCanvas,One Size
4,a327ffcf373d56860d2f95ee58c33256ae3516d087bf2e...,Bubble Split Midaxi Skirt,Boohoo (UK),Boohoo,Value,Women,Bottoms,Grey,Non-activewear,Plain,...,50.0%,12 Jul 2024,5/6,0,NaN,NaN,aw24,Sleek midi skirt with a daring side slit\nHigh...,"100% Polyester, Embroidery 100% Viscose","6, 8, 10, 12, 14, 16"



Test Data Head:


,Product ID,Name,Retailer,Brand,Segment,Gender,Category,Color,Activewear,Pattern,...,Current Discount Percentage,Date First Discounted,SKUs Available,Num Replenishments,Days to Majority SKU sellout,Days to First sellout,Season,Description,Care information,Sizes
0,1fd240b318ed3880dc4e15b9212f6410a01b06ac567462...,Yours Curve Black Linen Button Through Midaxi ...,SuperDrug (UK),Super Drug,Value,Women,Dresses,Black,Non-activewear,Plain,...,0.0%,NaN,1/1,0,NaN,NaN,aw25,NaN,NaN,20
1,5385cc61e6e5a1ea61d8f2ad556e0b2a2647b8ae5666e9...,PixieGirl Petite Pink Jacquard Side Stripe Wid...,SuperDrug (UK),Super Drug,Value,Women,Bottoms,Pink,Non-activewear,Stripes,...,0.0%,NaN,1/1,0,NaN,NaN,aw25,NaN,NaN,10
2,52d69007a22e64be803de380da0fc0dc2fd2ce51216707...,Yours Curve Black Tropical Floral Print Croppe...,SuperDrug (UK),Super Drug,Value,Women,Bottoms,Multicolour,Non-activewear,Floral,...,0.0%,NaN,1/1,0,NaN,NaN,aw25,NaN,NaN,16
3,e03515940ce7118dec7d2c9dc7fabde197596434d4aa50...,Blue Vanilla Brown Abstract Cow-print Shirt,SuperDrug (UK),Super Drug,Value,Women,Tops,Brown,Non-activewear,Abstract,...,0.0%,NaN,1/1,0,NaN,NaN,aw25,Change up your leopard print this season with ...,"Main: 80% Viscose, 20% Polyester",One Size
4,89c955b901bd4dddbb77dcc55102274cc0fdc259fbc3e3...,Where's That From Silver Dream Strappy Flat Sa...,SuperDrug (UK),Super Drug,Value,Women,Footwear,Silver,Non-activewear,Product has no pattern data,...,0.0%,NaN,1/1,0,NaN,NaN,aw25,"Brush the shoes with a brush to remove dust, d...",Faux Leather,UK3



Training Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 23 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Product ID                      60000 non-null  object 
 1   Name                            60000 non-null  object 
 2   Retailer                        60000 non-null  object 
 3   Brand                           60000 non-null  object 
 4   Segment                         60000 non-null  object 
 5   Gender                          60000 non-null  object 
 6   Category                        60000 non-null  object 
 7   Color                           60000 non-null  object 
 8   Activewear                      60000 non-null  object 
 9   Pattern                         60000 non-null  object 
 10  Full Price ($)                  60000 non-null  object 
 11  Original Currency               60000 non-null  object 
 12  Full Price 

### Clean Data

In [7]:
def clean_data(df):
    """Cleans and preprocesses the DataFrame."""
    df.columns = df.columns.str.strip()

    # Fill missing categorical values with the mode
    for col in ['Segment', 'Gender', 'Category', 'Color', 'Activewear', 'Pattern', 'Season']:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mode()[0])

    # Clean and convert numerical columns
    for col in ['Full Price ($)', 'Full Price (original currency)']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False), errors='coerce')
            df[col] = df[col].fillna(df[col].mean())

    if 'Current Discount Percentage' in df.columns:
        df['Current Discount Percentage'] = pd.to_numeric(df['Current Discount Percentage'].astype(str).str.replace('%', '', regex=False), errors='coerce')
        df['Current Discount Percentage'] = df['Current Discount Percentage'].fillna(df['Current Discount Percentage'].mean())

    # Fill missing numerical values with the mean
    for col in ['Num Replenishments', 'Days to Majority SKU sellout', 'Days to First sellout']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df[col] = df[col].fillna(df[col].mean())

    # Fill missing text values with 'Not available'
    for col in ['Name', 'Description', 'Care information', 'Sizes']:
        if col in df.columns:
            df[col] = df[col].fillna('Not available')

    # Convert 'Date First Discounted' to datetime
    if 'Date First Discounted' in df.columns:
        df['Date First Discounted'] = pd.to_datetime(df['Date First Discounted'], errors='coerce')

    return df

train_df = clean_data(train_df.copy())
test_df = clean_data(test_df.copy())

print("\nData cleaning complete.")
print("\nCleaned Training Data Info:")
train_df.info()


Data cleaning complete.

Cleaned Training Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 23 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Product ID                      60000 non-null  object        
 1   Name                            60000 non-null  object        
 2   Retailer                        60000 non-null  object        
 3   Brand                           60000 non-null  object        
 4   Segment                         60000 non-null  object        
 5   Gender                          60000 non-null  object        
 6   Category                        60000 non-null  object        
 7   Color                           60000 non-null  object        
 8   Activewear                      60000 non-null  object        
 9   Pattern                         60000 non-null  object        
 10  Full Price ($)  

 ### Feature Engineering and Target Creation

In [9]:
# NOTE ON DATA LEAKAGE: In a real-world scenario, 'return_risk' would be a label
# from historical data (e.g., a high return rate for a product). For this project,
# we are simulating this label based on feature values, which creates a data
# leakage issue leading to very high and unrealistic model performance.
# This approach is for demonstration purposes only.

def assign_return_risk(row):
    """Assigns a simulated return risk label based on business rules."""
    # High Risk
    if row['Category'] in ['Dresses', 'Jumpsuits & Playsuits'] and any(keyword in str(row['Name']).lower() for keyword in ['sequin', 'beaded', 'lace', 'bodycon', 'plunge', 'backless']):
        return 'High'
    if row['Full Price ($)'] > 200:
        return 'High'

    # Medium Risk
    if row['Category'] in ['Tops', 'Bottoms', 'Knitwear']:
        return 'Medium'
    if row['Current Discount Percentage'] > 20:
        return 'Medium'

    # Low Risk
    if row['Category'] in ['Accessories', 'Shoes', 'Bags']:
        return 'Low'
    if 'cotton' in str(row['Care information']).lower():
        return 'Low'

    return 'Low'

# Apply the logic to create the target variable
train_df['return_risk'] = train_df.apply(assign_return_risk, axis=1)
test_df['return_risk'] = test_df.apply(assign_return_risk, axis=1)

print("Return Risk Distribution in Training Data:")
display(train_df['return_risk'].value_counts())
print("\nReturn Risk Distribution in Test Data:")
display(test_df['return_risk'].value_counts())

# Combine text columns and create TF-IDF features
train_df['text'] = train_df['Name'] + ' ' + train_df['Description'] + ' ' + train_df['Care information']
test_df['text'] = test_df['Name'] + ' ' + test_df['Description'] + ' ' + test_df['Care information']

# Create and fit the TF-IDF vectorizer on the training text data
vectorizer = TfidfVectorizer(stop_words='english', max_features=100)
X_train_text = vectorizer.fit_transform(train_df['text'])
X_test_text = vectorizer.transform(test_df['text'])

# Convert to DataFrames for concatenation
X_train_text_df = pd.DataFrame(X_train_text.toarray(), columns=vectorizer.get_feature_names_out())
X_test_text_df = pd.DataFrame(X_test_text.toarray(), columns=vectorizer.get_feature_names_out())

# Define features and target
categorical_features = ['Segment', 'Category', 'Color']
numerical_features = ['Full Price ($)', 'Current Discount Percentage']
text_features = X_train_text_df.columns

# Combine all features
X = pd.concat([train_df[categorical_features], train_df[numerical_features], X_train_text_df], axis=1)
y = train_df['return_risk']

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print("\nShape of X_train:", X_train.shape)
print("Shape of X_val:", X_val.shape)

Return Risk Distribution in Training Data:


return_risk
Medium    24505
High      20634
Low       14861
Name: count, dtype: int64


Return Risk Distribution in Test Data:


return_risk
Low       22747
High      21101
Medium    16152
Name: count, dtype: int64


Shape of X_train: (48000, 105)
Shape of X_val: (12000, 105)


### Model Training and Evaluation Utilities

In [10]:
# Define the preprocessor once
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', StandardScaler(), numerical_features)
    ],
    remainder='passthrough'
)

def evaluate_model(model_pipeline, X_data, y_true, model_name):
    """Evaluates a trained model and prints key metrics."""
    y_pred = model_pipeline.predict(X_data)
    print(f"--- {model_name} Model Evaluation ---")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

def save_model(model, filename):
    """Saves a model to the trained_models directory."""
    if not os.path.exists('trained_models'):
        os.makedirs('trained_models')
    joblib.dump(model, f'trained_models/{filename}.joblib')
    print(f"\n{filename} saved successfully to 'trained_models'")

### Train and Evaluate Logistic Regression

In [11]:
print("Starting Logistic Regression training...")
logreg_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, solver='liblinear'))
])

logreg_model.fit(X_train, y_train)
evaluate_model(logreg_model, X_val, y_val, "Logistic Regression")

Starting Logistic Regression training...


C:\Users\ADMIN\return_risk\gtv\Lib\site-packages\sklearn\linear_model\_logistic.py:1288: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


--- Logistic Regression Model Evaluation ---
Accuracy: 0.8710833333333333

Classification Report:
              precision    recall  f1-score   support

        High       0.85      0.79      0.82      4137
         Low       0.86      0.92      0.89      2956
      Medium       0.89      0.91      0.90      4907

    accuracy                           0.87     12000
   macro avg       0.87      0.87      0.87     12000
weighted avg       0.87      0.87      0.87     12000


Confusion Matrix:
[[3266  379  492]
 [ 196 2706   54]
 [ 380   46 4481]]


### Train and Evaluate Random Forest

In [12]:
print("\nStarting Random Forest training...")
# Define a baseline model for comparison
rf_baseline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

rf_baseline.fit(X_train, y_train)
evaluate_model(rf_baseline, X_val, y_val, "Baseline Random Forest")
save_model(rf_baseline, 'random_forest_baseline')

# Hyperparameter tuning with RandomizedSearchCV
print("\nStarting Random Forest hyperparameter tuning...")
param_grid_rf = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [10, 20, 30, None],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__bootstrap': [True, False]
}

rf_random = RandomizedSearchCV(estimator=rf_baseline, param_distributions=param_grid_rf, n_iter=10, cv=3, verbose=1, random_state=42, n_jobs=-1)
rf_random.fit(X_train, y_train)

best_rf_model = rf_random.best_estimator_
print("\nBest parameters found for Random Forest:")
pprint(rf_random.best_params_)
evaluate_model(best_rf_model, X_val, y_val, "Tuned Random Forest")
save_model(best_rf_model, 'tuned_random_forest')



Starting Random Forest training...
--- Baseline Random Forest Model Evaluation ---
Accuracy: 0.9920833333333333

Classification Report:
              precision    recall  f1-score   support

        High       1.00      0.98      0.99      4137
         Low       0.98      1.00      0.99      2956
      Medium       0.99      1.00      1.00      4907

    accuracy                           0.99     12000
   macro avg       0.99      0.99      0.99     12000
weighted avg       0.99      0.99      0.99     12000


Confusion Matrix:
[[4044   51   42]
 [   2 2954    0]
 [   0    0 4907]]

random_forest_baseline saved successfully to 'trained_models/'

Starting Random Forest hyperparameter tuning...
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best parameters found for Random Forest:
{'classifier__bootstrap': False,
 'classifier__max_depth': None,
 'classifier__min_samples_leaf': 2,
 'classifier__min_samples_split': 5,
 'classifier__n_estimators': 100}
--- Tuned Random For

### Train and Evaluate XGBoost

In [13]:
print("\nStarting XGBoost training...")
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)

# Define a baseline model
xgb_baseline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(random_state=42))
])

xgb_baseline.fit(X_train, y_train_encoded)
evaluate_model(xgb_baseline, X_val, y_val_encoded, "Baseline XGBoost")
save_model(xgb_baseline, 'xgboost_baseline')

# Hyperparameter tuning with RandomizedSearchCV
print("\nStarting XGBoost hyperparameter tuning...")
param_grid_xgb = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3, 5, 7, 10],
    'classifier__learning_rate': [0.05, 0.1, 0.2],
    'classifier__subsample': [0.7, 0.8, 0.9],
    'classifier__colsample_bytree': [0.7, 0.8, 0.9]
}

xgb_random = RandomizedSearchCV(estimator=xgb_baseline, param_distributions=param_grid_xgb, n_iter=10, cv=3, verbose=1, random_state=42, n_jobs=-1)
xgb_random.fit(X_train, y_train_encoded)

best_xgb_model = xgb_random.best_estimator_
print("\nBest parameters found for XGBoost:")
pprint(xgb_random.best_params_)
evaluate_model(best_xgb_model, X_val, y_val_encoded, "Tuned XGBoost")
save_model(best_xgb_model, 'tuned_xgboost')


Starting XGBoost training...
--- Baseline XGBoost Model Evaluation ---
Accuracy: 0.99175

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      4137
           1       0.98      1.00      0.99      2956
           2       0.99      1.00      0.99      4907

    accuracy                           0.99     12000
   macro avg       0.99      0.99      0.99     12000
weighted avg       0.99      0.99      0.99     12000


Confusion Matrix:
[[4064   38   35]
 [  10 2945    1]
 [   4   11 4892]]

xgboost_baseline saved successfully to 'trained_models/'

Starting XGBoost hyperparameter tuning...
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best parameters found for XGBoost:
{'classifier__colsample_bytree': 0.8,
 'classifier__learning_rate': 0.05,
 'classifier__max_depth': 10,
 'classifier__n_estimators': 100,
 'classifier__subsample': 0.9}
--- Tuned XGBoost Model Evaluation ---
Accuracy: 0.9920833333333

### Final Evaluation on Test Data

In [14]:
print("  Final Evaluation on Test Data with Best Models  ")

# Prepare the test data
X_test = pd.concat([test_df[categorical_features], test_df[numerical_features], X_test_text_df], axis=1)
y_test = test_df['return_risk']
y_test_encoded = le.transform(y_test)

# 1. Evaluate the Tuned Random Forest model
print("\nRunning final evaluation with the Tuned Random Forest model...")
evaluate_model(best_rf_model, X_test, y_test, "Final Random Forest (on Test Data)")

# 2. Evaluate the Tuned XGBoost model
print("\nRunning final evaluation with the Tuned XGBoost model...")
y_pred_test_xgb = best_xgb_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test_encoded, y_pred_test_xgb))
print("\nClassification Report:")
print(classification_report(y_test_encoded, y_pred_test_xgb, target_names=le.classes_))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_encoded, y_pred_test_xgb))

  Final Evaluation on Test Data with Best Models  

Running final evaluation with the Tuned Random Forest model...
--- Final Random Forest (on Test Data) Model Evaluation ---
Accuracy: 0.9949333333333333

Classification Report:
              precision    recall  f1-score   support

        High       1.00      0.99      0.99     21101
         Low       0.99      1.00      0.99     22747
      Medium       1.00      1.00      1.00     16152

    accuracy                           0.99     60000
   macro avg       1.00      1.00      1.00     60000
weighted avg       0.99      0.99      0.99     60000


Confusion Matrix:
[[20802   296     3]
 [    5 22742     0]
 [    0     0 16152]]

Running final evaluation with the Tuned XGBoost model...
Accuracy: 0.99475

Classification Report:
              precision    recall  f1-score   support

        High       1.00      0.99      0.99     21101
         Low       0.99      1.00      0.99     22747
      Medium       1.00      1.00      1.00  